# 05 · Encoders em português: BERTimbau e Albertina

Ajuste fino com os rótulos do LLM, seleção pela validação do gabarito e avaliação no teste.

Ambiente: **GPU T4**.

In [ ]:
# Configuração: no Colab, clona o repositório e instala o pacote `absa`
import sys
if "google.colab" in sys.modules:
    !git clone -q https://github.com/vnevesl/pln-absa-olist.git 2>/dev/null || git -C pln-absa-olist pull -q
    %cd pln-absa-olist
    !pip -q install -e ".[transformers]"
    # Guarda cache do LLM e modelos no Drive para sobreviver à queda da sessão
    from google.colab import drive
    drive.mount("/content/drive")
    !mkdir -p /content/drive/MyDrive/pln-absa-olist/llm_cache /content/drive/MyDrive/pln-absa-olist/models
    !ln -sfn /content/drive/MyDrive/pln-absa-olist/llm_cache data/llm_cache
    !ln -sfn /content/drive/MyDrive/pln-absa-olist/models models

In [ ]:
import pandas as pd
from absa.config import ANNOTATED, INTERIM
from absa.metrics import bootstrap_ci, per_aspect_report, tuple_f1
from absa.models.encoder import ENCODERS, EncoderAspectClassifier, TrainConfig
from absa.preprocess import clean_light
from absa.sampling import load_gold

dev, test = load_gold()
fracos = pd.read_csv(INTERIM / "rotulos_llm.csv")
X_train, X_dev, X_test = (s.map(clean_light).tolist() for s in (fracos.texto, dev.texto, test.texto))

In [ ]:
resultados = {}
for nome in ["bertimbau", "albertina"]:
    cfg = TrainConfig(model_name=ENCODERS[nome], epochs=4, lr=3e-5, batch_size=32)
    clf = EncoderAspectClassifier(cfg)
    clf.fit(X_train, fracos, X_dev, dev, save_path=f"models/{nome}.pt")
    clf.load(f"models/{nome}.pt")  # melhor época na validação
    pred = clf.predict(X_test)
    resultados[nome] = pred
    f1, lo, hi = bootstrap_ci(test, pred)
    print(f"{nome}: F1 macro {f1:.3f} (IC95% {lo:.3f}–{hi:.3f}) | F1 tupla {tuple_f1(test, pred):.3f}")
    pred.assign(review_id=test.review_id.values).to_csv(INTERIM / f"pred_teste_{nome}.csv", index=False)

In [ ]:
per_aspect_report(test, resultados["bertimbau"]).round(3)